# Pilot Analysis (Tuần 7) — MS · Nhóm 2

Mục 7.3 RBL-4: (1) tính metric trên pilot, (2) vẽ histogram phân phối, (3) **xác nhận lựa chọn statistical test**.

> Sửa `INPUT` cho trỏ tới `metrics.csv` của pilot. Mặc định dùng data giả để chạy thử.

In [ ]:
import os, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy import stats

INPUT = "data/metrics.sample.csv"   # <-- doi thanh results/pilot/metrics.csv
GPT = "gpt4o-mini"
df = pd.read_csv(INPUT)
df["compiled"] = df["compiled"].astype(int)
df.loc[df["compiled"]==0, ["branch_coverage","mutation_score"]] = 0.0   # proposal §5.1 b4
gpt = df[df["method"]==GPT]
print("N hàm GPT:", gpt["function_id"].nunique(), "| invalid rate:",
      round((gpt["compiled"]==0).mean()*100,1), "%")

In [ ]:
# 1) Descriptive stats
display(df.groupby(["method","language"])[["branch_coverage","mutation_score"]].describe().round(1))

In [ ]:
# 2) Histogram phan phoi (GPT)
os.makedirs("results/figures", exist_ok=True)
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].hist(gpt["branch_coverage"].dropna(), bins=10); ax[0].axvline(80, color="r", ls="--", label="thr 80%")
ax[0].set_title("Branch Coverage"); ax[0].legend()
ax[1].hist(gpt["mutation_score"].dropna(), bins=10); ax[1].axvline(60, color="r", ls="--", label="thr 60%")
ax[1].set_title("Mutation Score"); ax[1].legend()
fig.tight_layout(); fig.savefig("results/figures/pilot_hist.png", dpi=300); plt.show()

In [ ]:
# 3) Xac nhan statistical test
x = gpt["branch_coverage"].dropna()
if len(x) >= 3:
    W, p = stats.shapiro(x)
    print(f"Shapiro normality p={p:.4f} ->", "KHONG chuan" if p<0.05 else "co the chuan")
print("=> Phan phoi metric thuong KHONG chuan => giu kiem dinh PHI THAM SO da chon trong proposal:")
print("   RQ1 One-sample Wilcoxon | RQ2 Paired Wilcoxon | RQ3 Spearman.")
print("Neu phan phoi RAT khac du kien (vd toan 0 do non-compile) -> ghi notes.md, bao PL, viet amendment (RBL-4 §7.3).")

## Kết luận pilot
- Pipeline đo chạy đúng, format `metrics.csv` hợp lệ → tiến hành full (Tuần 8).
- Test phi tham số phù hợp → **giữ nguyên** lựa chọn trong proposal (không HARKing).
- Nếu invalid rate > 20% hoặc phân phối bất thường → báo PL/GV trước khi scale.